In [2]:
import pandas as pd
# from sympy.physics.units import atomic_mass_unit

In [6]:
def convert_rule(r):
    # Helper function to format a single triple
    def format_triple(subj, pred, obj):
        #property_name = pred.split('/')[-1]
        #return f"{property_name}({subj.replace('?','')},{obj.replace('?','')})"
        return f"{pred}({subj.replace('?','')},{obj.replace('?','')})"
    body_str, head_str = r.split('=>')

    head_tokens = head_str.strip().split()

    if len(head_tokens) % 3 == 0:
        for i in range(0, len(head_tokens), 3):
            hsubj, pred, hobj = head_tokens[i:i+3]
            head_formatted = format_triple(hsubj, pred, hobj)

    # Process the body
    body_tokens = body_str.strip().split()

    body_formatted = []
    if len(body_tokens) % 3 == 0:
        for i in range(0, len(body_tokens), 3):
            subj, pred, obj = body_tokens[i:i+3]
            body_formatted.append(format_triple(subj, pred, obj))

    body_result = ", ".join(body_formatted)

    return f"{head_formatted} <= {body_result}"

def convert( amie_rules_path, outfile):
    amie_rules = pd.read_csv(amie_rules_path, sep='\t', header=None, skiprows=[0,1,2,-3,-2,-1],
                         names=['Rule','Head Coverage','Std Confidence','PCA Confidence','Positive Examples','Body size','PCA Body size','Functional variable'])
    print(len(amie_rules))
    amie_rules = amie_rules[(amie_rules['Body size'] >1) & (amie_rules['Std Confidence'] <1.0)]
    print(len(amie_rules))
    limited_rules = amie_rules.sort_values(by= 'Std Confidence', ascending=False)

    with open(outfile, 'w') as outfile:
        for i,row in limited_rules.iterrows():

            rule = convert_rule(row['Rule'])
            outfile.write(str(row['Body size']) +'\t'+ str(row['Positive Examples']) + '\t' + str(row['Std Confidence']) + '\t' + rule + '\n')

dataset= 'CSKG2'
rtype = '4CP'
amie_raw_rules_path=  f'../data/{dataset}/rules/{dataset}_{rtype}_rules.tsv'
amie_mined_rules_aligned_path= f'../data/{dataset}/rules/{dataset}_rules_amie_{rtype}.tsv'

convert(amie_raw_rules_path, amie_mined_rules_aligned_path)

0
0
